[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/rainbow/rainbow.ipynb)

# The Rainbow

This example traces sunlight refracted and reflected through a spherical raindrop to compute the primary and secondary rainbows on the sky. A parallel beam entering a spherical drop bends at its sunlit surface, bounces once or twice off the back rim, and bends again into air at a viewing angle controlled by its transverse impact offset from the drop centre. Because that viewing angle reaches a stationary extreme near 42 degrees from the antisolar point after one internal bounce and near 51 degrees after two, rays pile up at two sharp caustic edges separated by Alexander's dark band, while dispersion in water spreads violet from red across each arc.

Planar spacetime represents optical ***wavevectors*** and unit interface normals as extrinsic vectors, while ray displacements from the drop centre, exit ***directions***, and the drop's rest-frame time direction are intrinsic antivectors. Reflecting an incident wavevector across a unit interface plane via `face >> wavevector` reverses only its normal part, so `(wavevector + (face >> wavevector)) / 2` isolates the continuous tangential-and-temporal wavevector across the boundary and the medium dispersion relation sets the transmitted normal wavenumber without angles. Because every chord of a circle meets the rim at equal angles at both ends, reflecting an entry point across the chord plane gives the next bounce point and powers of a single centre rotor `turn` step every vertex and internal wavevector around the drop, while the normal wavenumbers `(wavevector | face)` give both Fresnel reflectances directly.

In the scalar and vector notation of geometric optics, refraction across a spherical interface requires tracking angles of incidence and refraction through Snell's law $n_1 \sin i = n_2 \sin r$ alongside a separate coplanarity condition $(\mathbf{k}_{\mathrm{in}} \times \mathbf{n}) \cdot \mathbf{k}_{\mathrm{out}} = 0$, accumulating total deflection $D_k(i) = 2(i - r) + k(\pi - 2r)$ with $r = \arcsin(\sin i / n)$ to locate Descartes's stationary caustic $\cos^2 i = (n^2 - 1) / (k(k + 2))$, and evaluating separate trigonometric Fresnel ratios $\sin^2(i - r) / \sin^2(i + r)$ and $\tan^2(i - r) / \tan^2(i + r)$ for perpendicular and in-plane polarization.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.optics.rainbow import render
from examples.util.animation import save_figure, show_states

np.set_printoptions(precision=2, suppress=True)

# Planar spacetime `x+y+t-` through the drop's cross-section, space positive and time negative.
ga = Algebra("x+y+t-")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Wavevector = ga.gatype.vector()                                               # Vector+
Plane = Wavevector.derive.within(Algebra("x+y+"))                             # Vector+
Event = ga.gatype.antivector()                                                # Antivector-
# Spatial displacements from the drop centre and ray directions in the drop rest frame.
Direction = Plane.derive.dual                                                 # Antivector-
Rotor = ga.gatype.rotor()                                                     # Rotor+

# Drop rest-frame time direction pairing to +1 with unit-frequency sunlight, and spatial axes.
time = -mv.t.dual()                                                           # [] Event-
forward, up = mv.x.dual(), mv.y.dual()                                        # [] Direction-, Direction-
sunlight_wave = mv.x - mv.t                                                   # [] Wavevector+

## Refraction and internal bounces in one raindrop

Rotational symmetry around the incoming sunbeam reduces a spherical raindrop to its planar cross-section across the impact offset `offsets`. Crossing the drop surface preserves the tangential spatial wavenumber and the temporal frequency while jumping the normal wavenumber to satisfy the dispersion relation of water, and reflecting the entry point across the refracted chord plane gives the fixed spatial rotor `turn` that carries every bounce around the rim.

In [ ]:
# Cauchy dispersion model across the visible spectrum in nanometres, and yellow index for the drop sweep.
n_colours = 31
wavelengths = np.linspace(400, 700, n_colours)
index_constant = 1.3238
index_dispersion = 3140.0
indices = (index_constant + index_dispersion / wavelengths**2)[:, None]       # [n_colours, 1]
drawn_index = np.array(1.333)
# Sample counts across the drop radius for sky integration, animated sweep, and background ray fan.
n_reflections = 2
n_rays = 4000
n_frames = 144
n_bundle = 48
frame_duration_ms = 40

In [ ]:
def refracted_wavevector(wavevector: Wavevector, face: Plane, index: np.ndarray) -> Wavevector:
    """Crossing a unit interface `face` (`[...] Plane+`) into a medium of refractive `index` preserves the
    tangential-and-temporal wavevector part and sets the normal wavenumber from dispersion (`[...] Wavevector+`)."""
    # `face >> wavevector` reverses the normal component while keeping the tangential spatial and temporal parts.
    tangential = (wavevector + (face >> wavevector)) / 2                      # [...] Wavevector+
    frequency = wavevector & time                                             # [...] Scalar+
    normal_wavenumber = ((index**2 - 1) * frequency.squared() - tangential.squared()).square_root()   # [...] Scalar+
    return tangential + face * normal_wavenumber                              # [...] Wavevector+


def entry_point(sunlight: Direction, offsets: Direction) -> Direction:
    """Rays travelling along unit `sunlight` (`[] Direction-`) at transverse `offsets` (`[...] Direction-`)
    meet the sunlit hemisphere of the unit drop at `entry_point` (`[...] Direction-`)."""
    return offsets - sunlight * (1 - offsets.antinorm().squared()).square_root()   # [...] Direction-

In [ ]:
def bounce_rotor(vertex: Direction, wavevector: Wavevector) -> Rotor:
    """Reflecting `vertex` (`[...] Direction-`) across the spatial chord along `wavevector` (`[...] Wavevector+`)
    defines the drop-centre rotor (`[...] Rotor+`) stepping each internal bounce to the next."""
    spatial = wavevector.cast(Plane)                                          # [...] Plane+
    chord_plane = spatial / spatial.norm()                                    # [...] Plane+
    # `chord_plane >> vertex` reverses the chordwise component and keeps the perpendicular offset from the centre.
    next_vertex = chord_plane >> vertex                                       # [...] Direction-
    return (next_vertex * vertex).square_root()                               # [...] Rotor+


def fresnel_reflectances(
    incident: Wavevector, refracted: Wavevector, face: Plane, index: np.ndarray,
) -> tuple[Scalar, Scalar]:
    """Normal wavenumbers of `incident` and `refracted` (`[...] Wavevector+`) across `face` (`[...] Plane+`)
    into refractive `index` determine the perpendicular and in-plane Fresnel reflectances (`[...] Scalar+` each)."""
    incident_normal = incident | face                                         # [...] Scalar+
    refracted_normal = refracted | face                                       # [...] Scalar+
    reflectance_across = ((incident_normal - refracted_normal) / (incident_normal + refracted_normal)).squared()   # [...] Scalar+
    reflectance_along = (
        (index**2 * incident_normal - refracted_normal) / (index**2 * incident_normal + refracted_normal)
    ).squared()                                                               # [...] Scalar+
    return reflectance_across, reflectance_along

Each internal reflection multiplies the transmitted fraction in both polarizations by the corresponding Fresnel reflectance, while equal-angle chord geometry keeps those reflectances identical at every bounce. An observer looking back against `exit_direction` measures the viewing angle from the antisolar direction `forward`, and rays entering at larger `offsets.antinorm()` intercept proportionally wider annular rings of sunlight.

In [ ]:
def drop_paths(
    offsets: Direction, index: np.ndarray, n_reflections: int,
) -> Iterator[tuple[Direction, Direction, Scalar]]:
    """Yields `(vertex, exit_direction, transmittance)` (`[...] Direction-`, `[...] Direction-`, `[...] Scalar+`)
    for zero up to `n_reflections` internal reflections of unit-frequency sunlight at impact `offsets`."""
    vertex = entry_point(forward, offsets)                                    # [...] Direction-
    inward_face = (-vertex).dual_inverse()                                    # [...] Plane+
    inside_wave = refracted_wavevector(sunlight_wave, inward_face, index)     # [...] Wavevector+
    # Equal chord angles around the circle let a single rotor step every internal bounce and wavevector.
    turn = bounce_rotor(vertex, inside_wave)                                  # [...] Rotor+
    reflectance_across, reflectance_along = fresnel_reflectances(sunlight_wave, inside_wave, inward_face, index)   # [...] Scalar+, Scalar+
    passed_across = (1 - reflectance_across).squared()                        # [...] Scalar+
    passed_along = (1 - reflectance_along).squared()                          # [...] Scalar+
    for _ in range(n_reflections + 1):
        vertex = turn >> vertex                                               # [...] Direction-
        exit_wave = refracted_wavevector(inside_wave, vertex.dual_inverse(), np.array(1.0))   # [...] Wavevector+
        exit_direction = exit_wave.cast(Plane).dual()                         # [...] Direction-
        yield vertex, exit_direction, (passed_across + passed_along) / 2
        inside_wave = turn >> inside_wave                                     # [...] Wavevector+
        passed_across = passed_across * reflectance_across                    # [...] Scalar+
        passed_along = passed_along * reflectance_along                       # [...] Scalar+


def sky_bows(
    offsets: Direction, indices: np.ndarray, n_reflections: int,
) -> tuple[Scalar, Scalar]:
    """Tracing `offsets` (`[n_rays] Direction-`) across `indices` (`[n_colours, 1]`) for `1 .. n_reflections`
    internal reflections gives the antisolar viewing cosines and ring-weighted fluxes (`[n_reflections, n_colours, n_rays] Scalar+` each)."""
    _, *bows = drop_paths(offsets, indices, n_reflections)
    # Looking back against `exit_direction` measures the angle from the antisolar direction `forward`.
    sight_cosines = stack([-(mv.x & exit_direction) for _, exit_direction, _ in bows])   # [..., ...] Scalar+
    ring_sizes = offsets.antinorm()                                           # [...] Scalar+
    return sight_cosines, stack([passed * ring_sizes for _, _, passed in bows])   # [..., ...] Scalar+

In [ ]:
def ray_bundle(
    offsets: Direction, index: np.ndarray, n_reflections: int,
) -> tuple[Direction, Direction, Direction]:
    """Entry points, bounce vertices, and exit directions (`[...] Direction-`, `[n_reflections + 1, ...] Direction-`,
    `[n_reflections + 1, ...] Direction-`) form the ray fan across `offsets`."""
    vertices, exits, _ = zip(*drop_paths(offsets, index, n_reflections))
    return entry_point(forward, offsets), stack(list(vertices)), stack(list(exits))


def swept_ray_states(
    offsets: Direction, index: np.ndarray, n_reflections: int,
) -> Iterator[tuple[Direction, Direction, Direction, Scalar]]:
    """Yields `(entry, vertices, exits, sight_cosines)` (`[] Direction-`, `[n_reflections + 1] Direction-`,
    `[n_reflections + 1] Direction-`, `[n_reflections + 1] Scalar+`) as a single ray steps across `offsets`."""
    for offset in offsets:
        entered, vertices, exits = ray_bundle(offset, index, n_reflections)
        yield entered, vertices, exits, -(mv.x & exits)


offsets = up * ((np.arange(n_rays) + 0.5) / n_rays)                           # [n_rays] Direction-
sight_cosines, flux = sky_bows(offsets, indices, n_reflections)               # [n_reflections, n_colours, n_rays] Scalar+, Scalar+
sweep_offsets = up * ((np.arange(n_frames) + 0.5) / n_frames)                 # [n_frames] Direction-
bundle_offsets = up * ((np.arange(n_bundle) + 0.5) / n_bundle)                # [n_bundle] Direction-
bundle = ray_bundle(bundle_offsets, drawn_index, n_reflections)
swept_states = show_states(
    swept_ray_states(sweep_offsets, drawn_index, n_reflections),
    render.sweep_animation,
    bundle,
    offsets,
    sight_cosines,
    wavelengths,
    name="rainbow_sweep",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

As the yellow incident ray climbs from the drop axis toward the rim on the left, the orange marker on the right rises to its stationary peak near 42 degrees and folds back while the blue double-bounce marker drops to its minimum near 51 degrees before climbing toward grazing incidence. In the faint background ray fan, outgoing orange and blue segments crowd densely along those two stationary envelopes and leave the angular band between 42 and 51 degrees empty.

## Primary and secondary bows on the sky

Every sunlit drop across the shower sends each wavelength back along a cone centred on the antisolar point below the horizon, so an observer on the ground sees concentric circular arcs clipped by the ground plane. Dividing the ring-weighted flux in each angular bin by the solid angle of that sky ring and convolving across the half-degree solar disc produces the sky radiance across all visible wavelengths.

In [ ]:
sun_elevation = 12.0
sun_width = 0.53
print("primary bow, violet to red (deg):  ", render.viewing_degrees(sight_cosines[0].min(axis=-1))[[0, -1]])
print("secondary bow, violet to red (deg):", render.viewing_degrees(sight_cosines[1].max(axis=-1))[[0, -1]])
save_figure(render.sky_figure(sight_cosines, flux, wavelengths, sun_elevation, sun_width), "rainbow_sky");

The primary arc places red on its outer rim near 42.3 degrees and violet on its inner edge near 40.5 degrees around a noticeably brighter interior sky where single-bounce rays fold back, while Alexander's dark band separates it from the fainter secondary arc with reversed colours from red near 50.4 degrees out to violet near 53.6 degrees.

In [ ]:
# checks
# Stationary deflection occurs at Descartes's incidence angle, largest at one bounce and smallest at two.
index = indices[:, 0]
reflections = np.arange(1, n_reflections + 1)[:, None]
incidence = np.arccos(np.sqrt((index**2 - 1) / (reflections * (reflections + 2))))
refraction = np.arcsin(np.sin(incidence) / index)
deflection = 2 * (incidence - refraction) + reflections * (np.pi - 2 * refraction)
assert (sight_cosines[0].min(axis=-1) + np.cos(deflection[0])).abs().max() < 1e-6
assert (sight_cosines[1].max(axis=-1) + np.cos(deflection[1])).abs().max() < 1e-6
# Every exited ray has unit spatial length, and secondary flux stays below primary flux at every offset.
exits = stack([state[2] for state in swept_states])                           # [n_frames, 3] Direction-
assert (exits.antinorm() - 1).abs().max() < 1e-6
assert (flux[1] < flux[0]).all()